# Chronic Kidney Disease (CKD) Prediction with Explainable AI (XAI)
### Enhanced Research with Model Interpretability
**Base Paper:** *Hassan et al. (2023), Human-Centric Intelligent Systems, 3:92–104*

**Enhancements:**
1. Original Pipeline (Data → Features → Models)
2. **XAI Module 1:** SHAP Values (Global & Local)
3. **XAI Module 2:** LIME (Model-Agnostic Explanations)
4. **XAI Module 3:** Permutation Feature Importance
5. **XAI Module 4:** Partial Dependence Plots
6. **XAI Module 5:** Individual Patient Prediction Explanations
7. **XAI Module 6:** Feature Interaction Analysis
8. Results Comparison & Clinical Insights

In [1]:
# Install required XAI packages
import sys
!{sys.executable} -m pip install shap lime -q
print("✓ XAI packages installed")

✓ XAI packages installed


  DEPRECATION: Building 'lime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'lime'. Discussion can be found at https://github.com/pypa/pip/issues/6334


## Phase 0: Library Imports & Configuration

In [4]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# Scikit-learn imports
from sklearn.experimental import enable_iterative_imputer  # Enable experimental features
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.base import clone
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import IterativeImputer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix, 
    classification_report, roc_auc_score, roc_curve, auc
)
from sklearn.inspection import permutation_importance, partial_dependence

# XAI imports
import shap
import lime
import lime.lime_tabular

# Boosting
import xgboost as xgb

# Visualization
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10
})

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## Phase 1: Data Loading & Preprocessing (Paper Methodology)

In [5]:
# Load data
df = pd.read_csv('kidney_disease.csv')

# Clean data
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].astype(str).str.strip()

df.replace(['?', 'None', 'nan'], np.nan, inplace=True)

print(f"Dataset Shape: {df.shape}")
print(f"\nMissing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nTarget Variable (classification) Distribution:")
print(df['classification'].value_counts())

Dataset Shape: (400, 26)

Missing Values:
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       70
wc       105
rc       130
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
dtype: int64

Target Variable (class) Distribution:


KeyError: 'class'

In [ ]:
# 1. Handle Missing Values using Iterative Imputation (PMM-like approach)
numeric_cols = df.select_dtypes(include=[np.number]).columns
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns

# Encode categorical columns
le_dict = {}
df_encoded = df.copy()

for col in non_numeric_cols:
    if col != 'classification':
        le = LabelEncoder()
        df_encoded[col] = pd.factorize(df[col])[0]
        le_dict[col] = le

# Encode target variable
le_target = LabelEncoder()
df_encoded['classification'] = le_target.fit_transform(df['classification'].astype(str))

# Impute missing values
imputer = IterativeImputer(max_iter=10, random_state=42)
df_imputed = pd.DataFrame(
    imputer.fit_transform(df_encoded),
    columns=df_encoded.columns
)

print("✓ Missing values handled using Iterative Imputation")
print(f"No missing values remaining: {df_imputed.isnull().sum().sum() == 0}")

## Phase 2: K-Means Clustering & XGBoost Feature Selection

In [ ]:
# 2. K-Means Clustering (Elbow Method)
X = df_imputed.drop('classification', axis=1)
y = df_imputed['classification']

# Find optimal k
inertias = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)

# Plot elbow
plt.figure(figsize=(10, 5))
plt.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='r', linestyle='--', label='Optimal k=3')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Elbow Method indicates k=3 as optimal")

In [ ]:
# 3. XGBoost Feature Selection using SHAP
# Train XGBoost for feature importance
xgb_model = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X, y)

# Get feature importance
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# Select top 7 features as per paper
top_features = importance_df.head(7)['feature'].tolist()

print("\n=== XGBoost Feature Importance ===")
print(importance_df.head(10))
print(f"\nTop 7 Selected Features:\n{top_features}")

## Phase 3: Train-Test Split & Model Training

In [ ]:
# Prepare datasets
X_full = X
X_selected = X[top_features]

# 80-20 split
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)

X_train_sel, X_test_sel, _, _ = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

scaler_sel = StandardScaler()
X_train_sel_scaled = scaler_sel.fit_transform(X_train_sel)
X_test_sel_scaled = scaler_sel.transform(X_test_sel)

print(f"Training set: {X_train_full_scaled.shape}")
print(f"Test set: {X_test_full_scaled.shape}")
print(f"✓ Data prepared for modeling")

In [ ]:
# Train all models
models = {
    'NN': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42),
    'RF': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'RT': DecisionTreeClassifier(random_state=42),
    'BTM': BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)
}

trained_models = {}
results = []

# Train on full dataset
for name, model in models.items():
    model.fit(X_train_full_scaled, y_train)
    trained_models[name] = {
        'model': model,
        'scaler': scaler_full,
        'X_train': X_train_full_scaled,
        'X_test': X_test_full_scaled,
        'dataset': 'Full (25 features)'
    }
    
    y_pred = model.predict(X_test_full_scaled)
    y_pred_proba = model.predict_proba(X_test_full_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': f"{acc*100:.2f}%",
        'Kappa': f"{kappa*100:.2f}%",
        'Dataset': 'Full'
    })
    
    print(f"✓ {name} trained on full dataset")

# Train on selected features
for name, model in models.items():
    model_copy = clone(model)
    model_copy.fit(X_train_sel_scaled, y_train)
    trained_models[f"{name}_SEL"] = {
        'model': model_copy,
        'scaler': scaler_sel,
        'X_train': X_train_sel_scaled,
        'X_test': X_test_sel_scaled,
        'dataset': 'Selected (7 features)'
    }
    
    y_pred = model_copy.predict(X_test_sel_scaled)
    acc = accuracy_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': f"{acc*100:.2f}%",
        'Kappa': f"{kappa*100:.2f}%",
        'Dataset': 'Selected'
    })
    
    print(f"✓ {name} trained on selected features")

results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)
print(results_df.to_string(index=False))

## Phase 4: XAI Module 1 - SHAP Values (Global Explanations)

In [ ]:
# Select best model (SVM on full dataset for this example)
best_model_name = 'SVM'
best_model_info = trained_models['SVM']
best_model = best_model_info['model']

# Create SHAP explainer
explainer = shap.KernelExplainer(
    model=lambda x: best_model.decision_function(x),
    data=X_train_full_scaled[:100],  # Use sample for speed
    link="identity"
)

# Calculate SHAP values for test set
shap_values = explainer.shap_values(X_test_full_scaled[:50])

print("✓ SHAP values calculated for SVM model")
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# SHAP Summary Plot (Bar)
plt.figure(figsize=(12, 6))
shap.summary_plot(
    shap_values,
    X_test_full_scaled[:50],
    feature_names=X_full.columns,
    plot_type='bar',
    show=False
)
plt.title('SHAP Summary Plot - Feature Importance for SVM', fontsize=14, fontweight='bold')
plt.xlabel('Mean |SHAP value| (average impact on model output)', fontsize=11)
plt.tight_layout()
plt.show()

print("✓ SHAP global feature importance computed")

In [ ]:
# SHAP Waterfall Plot (example prediction)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test_full_scaled[0],
        feature_names=X_full.columns
    ),
    show=False
)
plt.title('SHAP Waterfall Plot - Individual Prediction #1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ SHAP local explanation (individual prediction) generated")

## Phase 5: XAI Module 2 - LIME (Local Interpretable Model-Agnostic Explanations)

In [ ]:
# Initialize LIME explainer
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_full_scaled,
    feature_names=X_full.columns,
    class_names=['Not CKD', 'CKD'],
    mode='classification',
    random_state=42
)

# Explain individual predictions
exp_lime_1 = explainer_lime.explain_instance(
    data_row=X_test_full_scaled[0],
    predict_fn=best_model.predict_proba,
    num_features=10
)

exp_lime_2 = explainer_lime.explain_instance(
    data_row=X_test_full_scaled[1],
    predict_fn=best_model.predict_proba,
    num_features=10
)

# Plot LIME explanations
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

exp_lime_1.show_in_notebook(show_table=True, predict_proba=False)
print("\n" + "="*70)
exp_lime_2.show_in_notebook(show_table=True, predict_proba=False)

print("✓ LIME explanations generated for sample patients")

## Phase 6: XAI Module 3 - Permutation Feature Importance

In [ ]:
# Calculate permutation importance
perm_importance = permutation_importance(
    best_model,
    X_test_full_scaled,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': perm_importance.importances_mean,
    'Std': perm_importance.importances_std
}).sort_values('Importance', ascending=False)

print("Permutation Feature Importance:")
print(perm_df.head(10))

# Plot
plt.figure(figsize=(10, 6))
plt.barh(perm_df['Feature'].head(10), perm_df['Importance'].head(10), 
         xerr=perm_df['Std'].head(10), alpha=0.7, color='steelblue')
plt.xlabel('Permutation Importance', fontsize=11)
plt.title('Permutation Feature Importance - SVM Model', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("✓ Permutation importance computed")

## Phase 7: XAI Module 4 - Partial Dependence Plots